In [4]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, when

spark = SparkSession.builder \
    .appName("Tugas6-ETL-Produksi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [11]:
#A EXTRACT
df_transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_produk = spark.read.json("tugas6_produk.json")
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

print("df_transaksi:", df_transaksi.count(), "baris")
df_transaksi.printSchema()
print("df_produk:", df_produk.count(), "baris")
df_produk.printSchema()
print("df_ulasan:", df_ulasan.count(), "baris")
df_ulasan.printSchema()

df_transaksi: 5000 baris
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

df_produk: 30 baris
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

df_ulasan: 3500 baris
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [13]:
#B TRANSFORM PENGGABUNGAN
df_gabung = df_transaksi.join(df_produk, on="product_id", how="inner")
df_gabung = df_gabung.join(df_ulasan, on="order_id", how="left")
df_gabung = df_gabung.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))

print("Jumlah baris setelah join:", df_gabung.count())
df_gabung.select("order_id", "product_id", "kategori", "unit_terjual", "harga",
                  "total_pendapatan", "rating").show(10)

Jumlah baris setelah join: 5000
+--------+----------+------------+------------+------+----------------+------+
|order_id|product_id|    kategori|unit_terjual| harga|total_pendapatan|rating|
+--------+----------+------------+------------+------+----------------+------+
|     TX0|        21|  Elektronik|           2| 25000|           50000|     1|
|     TX1|         8|     Fashion|           6|250000|         1500000|  NULL|
|     TX2|        25|  Elektronik|           4| 25000|          100000|     2|
|     TX3|         3|     Fashion|           4| 75000|          300000|     5|
|     TX4|        19|Rumah Tangga|           6| 75000|          450000|  NULL|
|     TX5|        10|  Elektronik|           3| 75000|          225000|     5|
|     TX6|        26|   Kesehatan|           6| 75000|          450000|     4|
|     TX7|         4|     Makanan|           1|150000|          150000|  NULL|
|     TX8|         7|  Elektronik|           2| 75000|          150000|     5|
|     TX9|        30

In [14]:
#C. TRANSFORM  PENANGGANAN DATA KOSONG
df_gabung = df_gabung.withColumn("ada_ulasan", col("rating").isNotNull())

jumlah_tanpa_ulasan = df_gabung.filter(col("ada_ulasan") == False).count()
print("Transaksi tanpa ulasan:", jumlah_tanpa_ulasan)

df_etl_final = df_gabung.na.fill({"rating": 0})

print("Contoh data setelah rating diisi 0:")
df_etl_final.select("order_id", "kategori", "rating", "ada_ulasan", "total_pendapatan").show(10)

Transaksi tanpa ulasan: 1500
Contoh data setelah rating diisi 0:
+--------+------------+------+----------+----------------+
|order_id|    kategori|rating|ada_ulasan|total_pendapatan|
+--------+------------+------+----------+----------------+
|     TX0|  Elektronik|     1|      true|           50000|
|     TX1|     Fashion|     0|     false|         1500000|
|     TX2|  Elektronik|     2|      true|          100000|
|     TX3|     Fashion|     5|      true|          300000|
|     TX4|Rumah Tangga|     0|     false|          450000|
|     TX5|  Elektronik|     5|      true|          225000|
|     TX6|   Kesehatan|     4|      true|          450000|
|     TX7|     Makanan|     0|     false|          150000|
|     TX8|  Elektronik|     5|      true|          150000|
|     TX9|  Elektronik|     0|     false|          250000|
+--------+------------+------+----------+----------------+
only showing top 10 rows



In [19]:
#D LOAD
username = "[Azimah]" 

!hdfs dfs -mkdir -p /user/Azimah/tugas6/hasil_etl

df_etl_final.write.mode("overwrite").partitionBy("kategori").parquet(
    f"hdfs://localhost:9000/user/Azimah/tugas6/hasil_etl"
)

print("Data berhasil disimpan ke HDFS. Struktur folder:")
!hdfs dfs -ls -R /user/Azimah/tugas6/hasil_etl

Data berhasil disimpan ke HDFS. Struktur folder:
-rw-r--r--   3 Azimah supergroup          0 2026-09-26 08:47 /user/Azimah/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - Azimah supergroup          0 2026-09-26 08:47 /user/Azimah/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 Azimah supergroup      16361 2026-09-26 08:47 /user/Azimah/tugas6/hasil_etl/kategori=Elektronik/part-00000-2b88e45d-39ca-4706-a18a-69cdee2f56cb.c000.snappy.parquet
drwxr-xr-x   - Azimah supergroup          0 2026-09-26 08:47 /user/Azimah/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 Azimah supergroup      11131 2026-09-26 08:47 /user/Azimah/tugas6/hasil_etl/kategori=Fashion/part-00000-2b88e45d-39ca-4706-a18a-69cdee2f56cb.c000.snappy.parquet
drwxr-xr-x   - Azimah supergroup          0 2026-09-26 08:47 /user/Azimah/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 Azimah supergroup      10389 2026-09-26 08:47 /user/Azimah/tugas6/hasil_etl/kategori=Kesehatan/part-00000-2b88e45d-39ca-4706-a18a-69cdee2f56cb.c000.snap

In [20]:
#INSIGHT AKHIR
df_insight = df_etl_final.groupBy("kategori").agg(
    count("order_id").alias("jumlah_transaksi"),
    avg(when(col("ada_ulasan") == True, 1).otherwise(0)).alias("persentase_ada_ulasan")
).orderBy("persentase_ada_ulasan")

print("Persentase transaksi dengan ulasan per kategori (terendah di atas):")
df_insight.show()

Persentase transaksi dengan ulasan per kategori (terendah di atas):
+------------+----------------+---------------------+
|    kategori|jumlah_transaksi|persentase_ada_ulasan|
+------------+----------------+---------------------+
|     Makanan|             536|   0.6884328358208955|
|   Kesehatan|             961|   0.6888657648283039|
|     Fashion|            1035|   0.7014492753623188|
|Rumah Tangga|             815|   0.7055214723926381|
|  Elektronik|            1653|   0.7065940713853599|
+------------+----------------+---------------------+



In [21]:
# Menyimpan dengan kompresi berbeda untuk membandingkan ukuran file
df_etl_final.write.mode("overwrite").option("compression", "snappy") \
    .partitionBy("kategori").parquet("hasil_etl_snappy")

df_etl_final.write.mode("overwrite").option("compression", "gzip") \
    .partitionBy("kategori").parquet("hasil_etl_gzip")

!du -sh hasil_etl_snappy hasil_etl_gzip

108K	hasil_etl_snappy
104K	hasil_etl_gzip


In [22]:
df_baca_lagi = spark.read.parquet(f"hdfs://localhost:9000/user/Azimah/tugas6/hasil_etl")
df_filter = df_baca_lagi.filter(col("kategori") == "Elektronik")
df_filter.explain(True)

== Parsed Logical Plan ==
'Filter ('kategori = Elektronik)
+- Relation [order_id#973,product_id#974,unit_terjual#975,tanggal#976,harga#977L,nama_produk#978,rating#979,total_pendapatan#980L,ada_ulasan#981,kategori#982] parquet

== Analyzed Logical Plan ==
order_id: string, product_id: int, unit_terjual: int, tanggal: timestamp, harga: bigint, nama_produk: string, rating: int, total_pendapatan: bigint, ada_ulasan: boolean, kategori: string
Filter (kategori#982 = Elektronik)
+- Relation [order_id#973,product_id#974,unit_terjual#975,tanggal#976,harga#977L,nama_produk#978,rating#979,total_pendapatan#980L,ada_ulasan#981,kategori#982] parquet

== Optimized Logical Plan ==
Filter (isnotnull(kategori#982) AND (kategori#982 = Elektronik))
+- Relation [order_id#973,product_id#974,unit_terjual#975,tanggal#976,harga#977L,nama_produk#978,rating#979,total_pendapatan#980L,ada_ulasan#981,kategori#982] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [order_id#973,product_id#974,unit_

In [23]:
df_etl_final.cache()   # menandai DataFrame untuk disimpan di memori setelah aksi pertama

print("Total baris:", df_etl_final.count())          # trigger pertama: dihitung & disimpan di cache
print("Total pendapatan:", df_etl_final.agg({"total_pendapatan": "sum"}).collect()[0][0])  # trigger kedua: ambil dari cache, bukan hitung ulang dari awal

df_etl_final.unpersist()  # bersihkan cache setelah tidak dipakai lagi

Total baris: 5000
Total pendapatan: 2222975000


DataFrame[order_id: string, product_id: int, unit_terjual: int, tanggal: timestamp, harga: bigint, kategori: string, nama_produk: string, rating: int, total_pendapatan: bigint, ada_ulasan: boolean]

In [25]:
print("Jumlah partisi sebelum:", df_etl_final.rdd.getNumPartitions())

df_etl_final.coalesce(2).write.mode("overwrite") \
    .partitionBy("kategori").parquet("hasil_etl_rapi")

print("Selesai ditulis dengan jumlah file lebih sedikit per partisi.")
!hdfs dfs -ls /user/Azimah/tugas6/hasil_etl_rapi/kategori=Elektronik 2>/dev/null || ls hasil_etl_rapi

Jumlah partisi sebelum: 1


Selesai ditulis dengan jumlah file lebih sedikit per partisi.
'kategori=Elektronik'  'kategori=Kesehatan'  'kategori=Rumah Tangga'
'kategori=Fashion'     'kategori=Makanan'     _SUCCESS


In [26]:
from pyspark.sql.functions import broadcast

df_gabung_optimal = df_transaksi.join(broadcast(df_produk), on="product_id", how="inner")
df_gabung_optimal.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [product_id#413, order_id#412, unit_terjual#414, tanggal#415, harga#428L, kategori#429, nama_produk#430]
   +- BroadcastHashJoin [cast(product_id#413 as bigint)], [product_id#431L], Inner, BuildRight, false
      :- Filter isnotnull(product_id#413)
      :  +- FileScan csv [order_id#412,product_id#413,unit_terjual#414,tanggal#415] Batched: false, DataFilters: [isnotnull(product_id#413)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/Azimah/tugas6_transaksi.csv], PartitionFilters: [], PushedFilters: [IsNotNull(product_id)], ReadSchema: struct<order_id:string,product_id:int,unit_terjual:int,tanggal:timestamp>
      +- BroadcastExchange HashedRelationBroadcastMode(List(input[3, bigint, false]),false), [plan_id=4247]
         +- Filter isnotnull(product_id#431L)
            +- FileScan json [harga#428L,kategori#429,nama_produk#430,product_id#431L] Batched: false, DataFilters: [isnotnull(product_id#431L)], For

In [28]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
